[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/EN/Course_Notebooks/chapter13_lecture_notebook.ipynb)

---

# Chapter 13: LPPL Models for Bubble Detection

**Course:** Time Series Analysis and Forecasting  
**Program:** Bachelor program, Faculty of Cybernetics, Statistics and Economic Informatics, Bucharest University of Economic Studies, Romania  
**Academic Year:** 2025-2026

---

## Learning Objectives

By the end of this chapter, you will be able to:

1. Understand financial bubbles as critical phenomena with super-exponential growth
2. Simulate the 2D Ising model and interpret herding behavior via the Metropolis algorithm
3. Explain phase transitions, susceptibility divergence, and the connection to market regimes
4. Understand scale invariance, discrete scale invariance, and log-periodic oscillations
5. Derive and interpret the LPPL equation and its components
6. Fit LPPL to real market data using partial linearization and differential evolution
7. Apply the 8 filter conditions to validate bubble signals
8. Construct and interpret the LPPLS Confidence Indicator

## Setup and Imports

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !pip install yfinance scipy statsmodels matplotlib numpy pandas -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from scipy.optimize import differential_evolution
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# ── Course color scheme ─────────────────────────────────
COLORS = {
    'blue':       '#1A3A6E',
    'red':        '#DC3545',
    'green':      '#2E7D32',
    'amber':      '#B5853F',
    'orange':     '#E6802E',
    'purple':     '#8E44AD',
    'gray':       '#666666',
    'medium_gray':'#808080',
    'light_blue': '#5B8BD4',
}

# ── Global plot style ───────────────────────────────────
plt.rcParams.update({
    'figure.figsize':    (12, 5),
    'font.size':         11,
    'axes.facecolor':    'none',
    'figure.facecolor':  'none',
    'axes.grid':         False,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    False,
})

print('Setup complete!')

---
## Section 1: Financial Bubbles and Super-Exponential Growth

A **financial bubble** is a period where asset prices deviate significantly from fundamental values, driven by positive feedback loops (herding, momentum, leverage).

The key mathematical signature is **super-exponential growth**: log-price grows faster than linearly in time.

- **Normal growth:** $\ln P(t) = a + bt$ (exponential price growth, constant rate)
- **Bubble growth:** $\ln P(t) = A + B(t_c - t)^m$ with $B < 0$, $0 < m < 1$

The growth rate $d\ln P/dt = m|B|(t_c - t)^{m-1} \to \infty$ as $t \to t_c$, meaning the bubble **accelerates** toward a critical time $t_c$.

This is fundamentally different from exponential growth (constant rate) and cannot be sustained indefinitely — the system must undergo a **regime change** at or near $t_c$.

In [ ]:
# ── 1.1 Super-exponential vs exponential growth ───────────
t = np.linspace(0, 0.98, 300)
tc = 1.0

# Normal exponential growth
r = 0.3
p_exp = 100 * np.exp(r * t)

# Super-exponential bubble growth
m = 0.5; B = -5; A = np.log(100) - B
p_bubble = np.exp(A + B * (tc - t)**m)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Price trajectories
ax = axes[0]
ax.plot(t, p_exp, color=COLORS['green'], linewidth=3, label='Normal Growth')
ax.plot(t, p_bubble, color=COLORS['red'], linewidth=3, label='Bubble Growth')
ax.axvline(tc, color=COLORS['medium_gray'], linestyle='--', linewidth=2, alpha=0.7)
ax.set_yscale('log')
ax.annotate('Critical Time $t_c$', xy=(tc, 500), fontsize=12, ha='center',
            color=COLORS['medium_gray'],
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none'))
ax.set_xlabel('Time')
ax.set_ylabel('Price (log scale)')
ax.set_title('Price Trajectories: Normal vs Bubble', fontweight='bold')
ax.set_xlim(0, 1.1)
ax.grid(True, alpha=0.3)

# Right: Growth rates
ax2 = axes[1]
growth_exp = np.ones_like(t) * r
growth_bubble = m * np.abs(B) * (tc - t)**(m - 1)
mask = t < 0.92
ax2.plot(t, growth_exp, color=COLORS['green'], linewidth=3, label='Normal: Constant')
ax2.plot(t[mask], growth_bubble[mask], color=COLORS['red'], linewidth=3, label='Bubble: Accelerating')
ax2.axvline(tc, color=COLORS['medium_gray'], linestyle='--', linewidth=2, alpha=0.7)
ax2.annotate(r'$\rightarrow \infty$', xy=(0.93, 8.5), fontsize=14,
             color=COLORS['red'], fontweight='bold')
ax2.set_xlabel('Time')
ax2.set_ylabel(r'Growth Rate $d\ln P/dt$')
ax2.set_title('Growth Rate: The Key Difference', fontweight='bold')
ax2.set_xlim(0, 1.1); ax2.set_ylim(0, 10)
ax2.grid(True, alpha=0.3)

handles = [
    plt.Line2D([0], [0], color=COLORS['green'], linewidth=3, label='Normal Growth (Exponential)'),
    plt.Line2D([0], [0], color=COLORS['red'],   linewidth=3, label='Bubble Growth (Super-exponential)'),
    plt.Line2D([0], [0], color=COLORS['medium_gray'], linestyle='--', linewidth=2, label='Critical Time $t_c$'),
]
fig.legend(handles=handles, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=12)
plt.tight_layout()
plt.subplots_adjust(bottom=0.18)
plt.show()

print('Left: Bubble price diverges near tc (finite-time singularity).')
print('Right: Normal growth has a constant rate; bubble growth rate diverges.')

### Demo 1: Detecting Super-Exponential Growth in Real Data

We test for super-exponential growth in real bubble data by fitting three models to log-price:

1. **Linear:** $\ln P = a + bt$ (pure exponential price growth)
2. **Quadratic:** $\ln P = a + bt + ct^2$ (mild super-exponential)
3. **Exponential:** $\ln P = a + be^{ct}$ (strong super-exponential)

If $R^2$ is substantially higher for the nonlinear models, this is evidence of super-exponential growth — the hallmark of a bubble regime.

In [ ]:
# ── Demo 1: Super-exponential growth detection on NASDAQ Dot-com ──
import statsmodels.api as sm
from scipy.optimize import curve_fit

nasdaq = yf.download('^IXIC', start='1997-01-01', end='2000-03-10', progress=False)
close = nasdaq['Close']
if isinstance(close, pd.DataFrame):
    close = close.iloc[:, 0]
close = close.dropna()

y = np.log(close.values.astype(float))
t_data = np.arange(len(y))

# Model 1: Linear
X_lin = sm.add_constant(t_data)
mod_lin = sm.OLS(y, X_lin).fit()

# Model 2: Quadratic
X_quad = sm.add_constant(np.column_stack([t_data, t_data**2]))
mod_quad = sm.OLS(y, X_quad).fit()

# Model 3: Exponential
def exp_model(t, a, b, c):
    return a + b * np.exp(c * t)

try:
    popt, _ = curve_fit(exp_model, t_data, y, p0=[y[0], 0.01, 0.003], maxfev=10000)
    y_exp = exp_model(t_data, *popt)
    ss_res = np.sum((y - y_exp)**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2_exp = 1 - ss_res / ss_tot
except Exception:
    y_exp = mod_quad.predict(X_quad)
    r2_exp = 0.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(t_data, y, color=COLORS['gray'], lw=1, alpha=0.6, label='Observed log-price')
axes[0].plot(t_data, mod_lin.predict(X_lin), color=COLORS['blue'], lw=2, ls='--',
             label=f'Linear ($R^2$={mod_lin.rsquared:.4f})')
axes[0].plot(t_data, mod_quad.predict(X_quad), color=COLORS['red'], lw=2, ls='-.',
             label=f'Quadratic ($R^2$={mod_quad.rsquared:.4f})')
axes[0].plot(t_data, y_exp, color=COLORS['green'], lw=2, ls=':',
             label=f'Exponential ($R^2$={r2_exp:.4f})')
axes[0].set_title('NASDAQ Dot-com: Super-Exponential Growth Test', fontweight='bold')
axes[0].set_xlabel('Trading day (t)')
axes[0].set_ylabel('Log-price')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)

models = ['Linear', 'Quadratic', 'Exponential']
r2_values = [mod_lin.rsquared, mod_quad.rsquared, r2_exp]
bar_colors = [COLORS['blue'], COLORS['red'], COLORS['green']]
bars = axes[1].bar(models, r2_values, color=bar_colors, alpha=0.8, width=0.5)
for bar, val in zip(bars, r2_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f'{val:.4f}', ha='center', fontweight='bold', fontsize=11)
axes[1].set_title('$R^2$ Comparison', fontweight='bold')
axes[1].set_ylabel('$R^2$')
axes[1].set_ylim(0.9, 1.001)

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print(f'Quadratic and exponential R² dominate linear => super-exponential growth confirmed.')

---
## Section 2: The Ising Model and Collective Behavior

The **2D Ising model** is a lattice of $N \times N$ spins $s_i \in \{-1, +1\}$ (sell/buy) with energy:

$$H = -J \sum_{\langle i,j \rangle} s_i s_j$$

where $J > 0$ favors alignment (herding). The temperature $T$ controls randomness:

| Regime | Spin behavior | Market analogy |
|--------|--------------|----------------|
| $T \ll T_c$ | Strong alignment (ordered) | Strong herding — bubble regime |
| $T = T_c$ | Critical — clusters of all sizes | Maximum instability — regime break |
| $T \gg T_c$ | Random (disordered) | Independent trading — efficient market |

The **critical temperature** is $T_c = \frac{2}{\ln(1 + \sqrt{2})} \approx 2.269$ (Onsager, 1944).

At $T_c$:
- Magnetization $|M| \to 0$ continuously (second-order phase transition)
- Susceptibility $\chi \to \infty$ (infinite sensitivity to perturbation)
- Correlation length $\xi \to \infty$ (market-wide contagion)

In [ ]:
# ── 2.1 Metropolis Algorithm with Vectorized Checkerboard Updates ──
def checkerboard_sweep(spins, beta, parity):
    """One sweep of checkerboard Metropolis updates."""
    n = spins.shape[0]
    rows, cols = np.meshgrid(np.arange(n), np.arange(n), indexing='ij')
    mask = (rows + cols) % 2 == parity
    nb = (np.roll(spins, 1, 0) + np.roll(spins, -1, 0) +
          np.roll(spins, 1, 1) + np.roll(spins, -1, 1))
    dE = 2.0 * spins * nb
    prob = np.exp(-beta * dE)
    flip = (dE <= 0) | (np.random.random((n, n)) < prob)
    spins[mask & flip] *= -1
    return spins


def equilibrate(spins, T, sweeps=3000):
    """Run Metropolis sweeps to reach thermal equilibrium."""
    if T < 0.01:
        return spins
    beta = 1.0 / T
    for _ in range(sweeps):
        checkerboard_sweep(spins, beta, 0)
        checkerboard_sweep(spins, beta, 1)
    return spins


N = 60
Tc = 2.0 / np.log(1.0 + np.sqrt(2.0))
cmap_ising = LinearSegmentedColormap.from_list('ising', [COLORS['red'], 'white', COLORS['blue']])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Low T — ordered (bubble regime)
np.random.seed(42)
low_T = np.ones((N, N), dtype=int)
low_T = equilibrate(low_T, 0.5 * Tc)
axes[0].imshow(low_T, cmap=cmap_ising, vmin=-1, vmax=1)
axes[0].set_title('$T < T_c$: Ordered\n$|m| \\approx 1$ (Strong consensus)', fontweight='bold')
axes[0].set_xticks([]); axes[0].set_yticks([])
axes[0].text(0.5, -0.12, 'Market: Strong herding\nBubble regime',
             transform=axes[0].transAxes, ha='center', fontsize=11, style='italic')

# Critical point
np.random.seed(42)
crit = np.ones((N, N), dtype=int)
crit = equilibrate(crit, Tc)
axes[1].imshow(crit, cmap=cmap_ising, vmin=-1, vmax=1)
axes[1].set_title('$T = T_c$: Critical Point\nClusters of ALL sizes!', fontweight='bold')
axes[1].set_xticks([]); axes[1].set_yticks([])
axes[1].text(0.5, -0.12, 'Market: Maximum instability\nSmall trigger → large cascade',
             transform=axes[1].transAxes, ha='center', fontsize=11, style='italic',
             color=COLORS['red'])

# High T — disordered (normal market)
np.random.seed(42)
high_T = np.ones((N, N), dtype=int)
high_T = equilibrate(high_T, 2.0 * Tc)
axes[2].imshow(high_T, cmap=cmap_ising, vmin=-1, vmax=1)
axes[2].set_title('$T > T_c$: Disordered\n$m \\approx 0$ (No consensus)', fontweight='bold')
axes[2].set_xticks([]); axes[2].set_yticks([])
axes[2].text(0.5, -0.12, 'Market: Random trading\nNo herding behavior',
             transform=axes[2].transAxes, ha='center', fontsize=11, style='italic')

handles = [
    mpatches.Patch(color=COLORS['blue'], label='Spin Up (+1) = BUY'),
    mpatches.Patch(color=COLORS['red'],  label='Spin Down (-1) = SELL'),
    mpatches.Patch(color='white', edgecolor='gray', label='Neutral'),
]
fig.legend(handles=handles, loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=12)
plt.tight_layout()
plt.subplots_adjust(bottom=0.18)
plt.show()

print(f'Lattice: {N}×{N}, Tc = {Tc:.3f}')
print(f'Magnetization: |M(low)| = {abs(low_T.mean()):.3f}, '
      f'|M(crit)| = {abs(crit.mean()):.3f}, |M(high)| = {abs(high_T.mean()):.3f}')

### Demo 2: Phase Transition — Magnetization and Susceptibility

The **order parameter** (magnetization) and **susceptibility** completely characterize the phase transition:

- **Magnetization** $|M| \sim (T_c - T)^\beta$ with $\beta = 1/8$ vanishes continuously at $T_c$
- **Susceptibility** $\chi \sim |T - T_c|^{-\gamma}$ with $\gamma = 7/4$ diverges at $T_c$

Financial interpretation: susceptibility $\chi \to \infty$ means the market becomes infinitely sensitive to small perturbations — a tiny piece of news can trigger a market-wide cascade.

In [ ]:
# ── Demo 2: Phase transition diagrams ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
T_range = np.linspace(0.01, 3, 500)
Tc_exact = 2.269

# Magnetization
ax = axes[0]
mag = np.where(T_range < Tc_exact, (1 - (T_range/Tc_exact)**2)**0.125, 0.0)
ax.plot(T_range, mag, color=COLORS['blue'], linewidth=3, label='Magnetization $|m|$')
ax.plot(T_range, -mag, color=COLORS['blue'], linewidth=3, alpha=0.5)
ax.axvline(Tc_exact, color=COLORS['red'], linestyle='--', linewidth=2.5,
           label=f'$T_c = {Tc_exact:.3f}$')
ax.fill_betweenx([0, 1], 0, Tc_exact, color=COLORS['green'], alpha=0.15,
                 label='Ordered Phase')
ax.fill_betweenx([0, 1], Tc_exact, 3, color=COLORS['orange'], alpha=0.15,
                 label='Disordered Phase')
ax.set_xlabel('Temperature $T$')
ax.set_ylabel('Magnetization $m$')
ax.set_title('(A) Order Parameter: Magnetization', fontweight='bold')
ax.set_xlim(0, 3); ax.set_ylim(-1.1, 1.1)
ax.grid(True, alpha=0.3)

# Susceptibility
ax = axes[1]
chi = np.where(np.abs(T_range - Tc_exact) > 0.05,
               1 / np.abs(T_range - Tc_exact)**1.75, np.nan)
chi = np.clip(chi, 0, 50)
ax.plot(T_range, chi, color=COLORS['red'], linewidth=3)
ax.axvline(Tc_exact, color=COLORS['medium_gray'], linestyle='--', linewidth=2)
ax.annotate(r'$\chi \to \infty$' + '\nat $T_c$!', xy=(Tc_exact, 40),
            fontsize=13, ha='center',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none'))
ax.set_xlabel('Temperature $T$')
ax.set_ylabel(r'Susceptibility $\chi$')
ax.set_title('(B) Susceptibility Diverges at Critical Point', fontweight='bold')
ax.set_xlim(0, 3); ax.set_ylim(0, 55)
ax.grid(True, alpha=0.3)

handles = [
    plt.Line2D([0], [0], color=COLORS['blue'], linewidth=3, label='Magnetization $|m|$'),
    plt.Line2D([0], [0], color=COLORS['red'],  linewidth=3, label=r'Susceptibility $\chi$'),
    mpatches.Patch(color=COLORS['green'], alpha=0.3, label='Ordered (Bubble)'),
    mpatches.Patch(color=COLORS['orange'], alpha=0.3, label='Disordered (Normal)'),
]
fig.legend(handles=handles, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.02), frameon=False, fontsize=11)
plt.tight_layout()
plt.subplots_adjust(bottom=0.14)
plt.show()

print('At Tc: magnetization vanishes, susceptibility diverges.')
print('Financial implication: maximum market fragility at the critical point.')

### Demo 3: Ising Simulation — Bubble Formation and Regime Break

We simulate the Ising model heating from $T/T_c = 0.50$ (strong herding) through the critical point to $T/T_c = 2.00$ (normal market). This visualizes:

- **Row 1 (Bubble formation):** Ordered clusters gradually fragment as temperature rises toward $T_c$
- **Row 2 (Recovery):** At and beyond $T_c$, large-scale order disappears — herding collapses

In [ ]:
# ── Demo 3: Ising simulation — Bubble and Recovery ─────────
cmap_bin = ListedColormap([COLORS['red'], COLORS['blue']])
SWEEPS = 3000

bubble_panels = [
    (0.50, 'Strong herding',    COLORS['blue']),
    (0.80, 'Fluctuations grow', COLORS['amber']),
    (0.94, 'Approaching $T_c$', COLORS['red']),
    (1.00, 'Regime break',      COLORS['red']),
]
crash_panels = [
    (1.00, 'Regime break',      COLORS['red']),
    (1.10, 'Herding collapses', COLORS['red']),
    (1.50, 'Approaching normal',COLORS['green']),
    (2.00, 'Normal market',     COLORS['green']),
]

# Generate bubble snapshots
np.random.seed(42)
spins_b = np.ones((N, N), dtype=int)
spins_b = equilibrate(spins_b, 0.50 * Tc, SWEEPS * 2)
bubble_snapshots = []
for ratio, _, _ in bubble_panels:
    spins_b = equilibrate(spins_b, ratio * Tc, SWEEPS)
    bubble_snapshots.append((spins_b.copy(), abs(spins_b.mean())))

# Generate crash snapshots
np.random.seed(123)
spins_c = np.ones((N, N), dtype=int)
spins_c = equilibrate(spins_c, 0.50 * Tc, SWEEPS * 2)
crash_snapshots = []
for ratio, _, _ in crash_panels:
    spins_c = equilibrate(spins_c, ratio * Tc, SWEEPS)
    crash_snapshots.append((spins_c.copy(), abs(spins_c.mean())))

# Build figure
fig = plt.figure(figsize=(14, 8))
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.35, wspace=0.15)

for col, ((snap, mag), (ratio, label, color)) in enumerate(
        zip(bubble_snapshots, bubble_panels)):
    ax = fig.add_subplot(gs[0, col])
    ax.imshow((snap + 1) // 2, cmap=cmap_bin, vmin=0, vmax=1, interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'$T/T_c = {ratio:.2f}$\n{label}', fontsize=9,
                 fontweight='bold', color=color, pad=4)
    ax.text(0.5, -0.08, f'$|M| = {mag:.2f}$', transform=ax.transAxes,
            ha='center', fontsize=8, color=COLORS['medium_gray'])

fig.text(0.02, 0.72, 'Bubble\nFormation', fontsize=11, fontweight='bold',
         color=COLORS['red'], ha='center', va='center', rotation=90)

for col, ((snap, mag), (ratio, label, color)) in enumerate(
        zip(crash_snapshots, crash_panels)):
    ax = fig.add_subplot(gs[1, col])
    ax.imshow((snap + 1) // 2, cmap=cmap_bin, vmin=0, vmax=1, interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'$T/T_c = {ratio:.2f}$\n{label}', fontsize=9,
                 fontweight='bold', color=color, pad=4)
    ax.text(0.5, -0.08, f'$|M| = {mag:.2f}$', transform=ax.transAxes,
            ha='center', fontsize=8, color=COLORS['medium_gray'])

fig.text(0.02, 0.30, 'Break &\nRecovery', fontsize=11, fontweight='bold',
         color=COLORS['green'], ha='center', va='center', rotation=90)

fig.text(0.5, 0.01,
         f'2D Ising model ({N}×{N}). Blue = buy, red = sell. '
         f'Top: bubble forms as herding strengthens approaching $T_c$. '
         f'Bottom: regime break at $T_c$, recovery above.',
         ha='center', fontsize=9, color=COLORS['medium_gray'])

fig.patch.set_alpha(0)
plt.show()

---
## Section 3: Scale Invariance and Log-Periodicity

### Continuous Scale Invariance
A function $f(x)$ is **scale-invariant** if $f(\lambda x) = \lambda^\alpha f(x)$ for all $\lambda > 0$. The only solution is a **power law**: $f(x) = C x^\alpha$.

### Discrete Scale Invariance (DSI)
If the scaling relation holds only for a **preferred ratio** $\lambda$, i.e., $f(\lambda x) = \lambda^\alpha f(x)$ for a specific $\lambda$, the general solution is:

$$f(x) = x^\alpha \left[ A + B \cos\!\left(\omega \ln x + \phi\right) \right]$$

where $\omega = 2\pi / \ln \lambda$ is the **log-frequency**.

This is the mathematical origin of **log-periodic oscillations** in the LPPL model. In financial markets, DSI arises from the **hierarchical structure** of traders — agents influence each other across multiple organizational scales.

In [ ]:
# ── 3.1 Log-periodic oscillations — the key signature ──────
t_lp = np.linspace(0, 0.95, 500)
tc_lp = 1.0; m_lp = 0.5; omega_lp = 8
dt = tc_lp - t_lp

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (A) Power law trend
ax = axes[0, 0]
ax.plot(t_lp, dt**m_lp, color=COLORS['blue'], linewidth=3)
ax.set_xlabel('Time $t$')
ax.set_ylabel('$(t_c - t)^m$')
ax.set_title('(A) Power Law Trend', fontweight='bold')
ax.grid(True, alpha=0.3)

# (B) Log-periodic oscillation
ax = axes[0, 1]
osc = np.cos(omega_lp * np.log(dt))
ax.plot(t_lp, osc, color=COLORS['green'], linewidth=2)
ax.set_xlabel('Time $t$')
ax.set_ylabel(r'$\cos(\omega \ln(t_c - t))$')
ax.set_title('(B) Log-Periodic Oscillation', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(-1.5, 1.5)

# (C) Combined LPPL (log scale)
ax = axes[1, 0]
combined = dt**m_lp * (1 + 0.3 * osc)
ax.plot(t_lp, combined, color=COLORS['red'], linewidth=2)
ax.set_xlabel('Time $t$')
ax.set_ylabel(r'$\ln P(t)$')
ax.set_title('(C) Combined LPPL (Log Scale)', fontweight='bold')
ax.grid(True, alpha=0.3)

# (D) In log-time: oscillations are periodic!
ax = axes[1, 1]
log_dt = np.log(dt)
ax.plot(log_dt, osc, color=COLORS['orange'], linewidth=2)
ax.set_xlabel(r'$\ln(t_c - t)$')
ax.set_ylabel('Oscillation')
ax.set_title('(D) In Log-Time: Oscillations Are Periodic!', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_ylim(-1.5, 1.5)
ax.annotate('Key insight:\nEvenly spaced\nin log-time!', xy=(-2, 0.8),
            fontsize=12, ha='center',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none'))

fig.text(0.5, 0.02,
         r'Log-periodic oscillations compress in real time but are evenly spaced in $\ln(t_c - t)$' + '\n'
         'This is the signature of Discrete Scale Invariance from hierarchical market structure',
         ha='center', fontsize=12, style='italic')
fig.patch.set_alpha(0)
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)
plt.show()

### Demo 4: The Preferred Scaling Ratio $\lambda$

The **scaling ratio** $\lambda = e^{2\pi/\omega}$ determines the self-similar structure of log-periodic oscillations. Each successive oscillation peak is spaced by a factor $\lambda$ in time-to-critical:

$$\Delta t_{n+1} / \Delta t_n = \lambda$$

Empirically, most financial bubbles show $\lambda \approx 2$, meaning each cycle of hesitation is roughly half as long as the previous one.

In [ ]:
# ── Demo 4: Preferred scaling ratio ────────────────────────
omega_vals = np.linspace(4, 25, 200)
lambda_vals = np.exp(2 * np.pi / omega_vals)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: lambda vs omega
ax = axes[0]
ax.plot(omega_vals, lambda_vals, color=COLORS['blue'], linewidth=3)
ax.axhline(2.0, color=COLORS['red'], linestyle='--', linewidth=2,
           label=r'$\lambda = 2$ (most common)')
omega_2 = 2 * np.pi / np.log(2)
ax.axvline(omega_2, color=COLORS['amber'], linestyle=':', linewidth=1.5,
           label=f'$\omega = 2\pi/\ln 2 \\approx {omega_2:.1f}$')
ax.set_xlabel(r'$\omega$ (log-frequency)')
ax.set_ylabel(r'$\lambda = e^{2\pi/\omega}$')
ax.set_title('Scaling Ratio vs Log-Frequency', fontweight='bold')
ax.set_xlim(4, 25); ax.set_ylim(1, 5)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False)

# Right: Geometric convergence of oscillation peaks
ax = axes[1]
tc_demo = 100
lam_demo = 2.0
n_peaks = 6
dt_peaks = [tc_demo * lam_demo**(-k) for k in range(n_peaks)]
t_peaks = [tc_demo - d for d in dt_peaks]

t_cont = np.linspace(0, tc_demo - 1, 500)
dt_cont = tc_demo - t_cont
lppl_demo = dt_cont**0.5 * (1 + 0.3 * np.cos(omega_2 * np.log(dt_cont)))
ax.plot(t_cont, lppl_demo, color=COLORS['blue'], linewidth=2)
for i, tp in enumerate(t_peaks):
    if 0 < tp < tc_demo - 1:
        ax.axvline(tp, color=COLORS['red'], linewidth=1, alpha=0.5)
        if i < 5:
            ax.annotate(f'$\\Delta t_{i+1}/\\Delta t_{i} = \\lambda$',
                        xy=(tp, 0.2), fontsize=7, color=COLORS['red'], rotation=90)
ax.set_xlabel('Time')
ax.set_ylabel('LPPL signal')
ax.set_title(r'Geometric Convergence: $\lambda = 2$', fontweight='bold')
ax.grid(True, alpha=0.3)

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print(f'For omega = {omega_2:.1f}: lambda = exp(2*pi/omega) = {np.exp(2*np.pi/omega_2):.3f}')
print('Each oscillation cycle is lambda times shorter than the previous one.')

---
## Section 4: The LPPL Model — Derivation and Components

The **Log-Periodic Power Law** (LPPL) model describes the log-price during a bubble:

$$\ln P(t) = A + B(t_c - t)^m + C(t_c - t)^m \cos\!\left(\omega \ln(t_c - t) - \varphi\right)$$

### Parameter Interpretation

| Parameter | Meaning | Financial Interpretation |
|-----------|---------|------------------------|
| $A$ | Log-price at $t_c$ | Where is the price heading? |
| $B < 0$ | Power-law amplitude | How fast is the bubble growing? |
| $m \in (0,1)$ | Power-law exponent | Degree of super-exponential growth |
| $C$ | Oscillation amplitude | Strength of trader hesitation |
| $\omega$ | Log-frequency | How many rounds of hesitation? |
| $\varphi$ | Phase shift | Timing of oscillation pattern |
| $t_c$ | Critical time | When does the music stop? |

### The Three Components

1. **Constant $A$:** The expected log-price at the critical time
2. **Power law $B(t_c - t)^m$:** Super-exponential acceleration ($B < 0$ ensures price *rises* as $t \to t_c$)
3. **Log-periodic $C(t_c-t)^m\cos(\omega\ln(t_c-t)-\varphi)$:** Oscillations = rounds of doubt among traders

In [ ]:
# ── 4.1 LPPL Components Decomposition ─────────────────────
t_comp = np.linspace(0, 340, 500)
tc_comp = 341
m_c = 0.8; omega_c = 6; A_c = 4; B_c = -0.02; C_c = 0.005; phi_c = 2
dt_comp = np.maximum(tc_comp - t_comp, 0.1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# (A) Constant A
ax = axes[0, 0]
ax.axhline(A_c, color=COLORS['blue'], linewidth=3)
ax.fill_between(t_comp, A_c - 0.1, A_c + 0.1, color=COLORS['blue'], alpha=0.2)
ax.set_ylabel('$A$'); ax.set_xlabel('Time')
ax.set_title('(A) Constant: $A$ = Log price at $t_c$', fontweight='bold')
ax.set_ylim(A_c - 1, A_c + 1)
ax.grid(True, alpha=0.3)

# (B) Power law
ax = axes[0, 1]
power = B_c * dt_comp**m_c
ax.plot(t_comp, power, color=COLORS['red'], linewidth=3)
ax.fill_between(t_comp, 0, power, color=COLORS['red'], alpha=0.2)
ax.set_ylabel('$B(t_c - t)^m$'); ax.set_xlabel('Time')
ax.set_title(f'(B) Power Law: $B(t_c - t)^m$, $m={m_c}$', fontweight='bold')
ax.grid(True, alpha=0.3)

# (C) Log-periodic
ax = axes[1, 0]
logper = C_c * dt_comp**m_c * np.cos(omega_c * np.log(dt_comp) - phi_c)
ax.plot(t_comp, logper, color=COLORS['green'], linewidth=2)
ax.fill_between(t_comp, 0, logper, color=COLORS['green'], alpha=0.2)
ax.set_ylabel(r'$C(t_c-t)^m\cos(...)$'); ax.set_xlabel('Time')
ax.set_title(f'(C) Log-Periodic: $\omega={omega_c}$', fontweight='bold')
ax.grid(True, alpha=0.3)

# (D) Full LPPL
ax = axes[1, 1]
lppl_full = A_c + B_c * dt_comp**m_c + C_c * dt_comp**m_c * np.cos(omega_c * np.log(dt_comp) - phi_c)
ax.plot(t_comp, np.exp(lppl_full), color=COLORS['blue'], linewidth=3,
        label=r'Price = $e^{\mathrm{LPPL}}$')
ax.axvline(tc_comp, color=COLORS['orange'], linestyle='--', linewidth=2, label='$t_c$')
ax.set_ylabel('Price'); ax.set_xlabel('Time')
ax.set_title('(D) Full LPPL Model', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.10), ncol=2, frameon=False)

fig.text(0.5, 0.02,
         r'$\ln P(t) = A + B(t_c - t)^m + C(t_c - t)^m \cos(\omega \ln(t_c - t) - \varphi)$',
         ha='center', fontsize=14,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='none'))
fig.patch.set_alpha(0)
plt.tight_layout()
plt.subplots_adjust(bottom=0.10)
plt.show()

### Demo 5: The Crash Hazard Rate

The LPPL model is derived from a **rational expectations** framework where the crash hazard rate is:

$$h(t) = \alpha (t_c - t)^{m-1} \left[1 + \beta \cos\!\left(\omega \ln(t_c - t) - \varphi\right)\right]$$

As $t \to t_c$, $h(t) \to \infty$: the crash becomes increasingly likely but is **never certain**. The log-periodic oscillations in $h(t)$ reflect alternating waves of confidence and fear among traders.

In [ ]:
# ── Demo 5: Crash hazard rate ──────────────────────────────
t_hz = np.linspace(0, 0.95, 500)
tc_hz = 1.0; m_hz = 0.5; omega_hz = 8; beta_hz = 0.3
dt_hz = tc_hz - t_hz

h_power = dt_hz**(m_hz - 1)
h_full = h_power * (1 + beta_hz * np.cos(omega_hz * np.log(dt_hz)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Hazard rate components
ax = axes[0]
mask_hz = t_hz < 0.92
ax.plot(t_hz[mask_hz], h_power[mask_hz], color=COLORS['blue'], linewidth=2,
        label=r'Power-law envelope $(t_c-t)^{m-1}$')
ax.plot(t_hz[mask_hz], h_full[mask_hz], color=COLORS['red'], linewidth=2,
        label=r'Full hazard rate $h(t)$')
ax.axvline(tc_hz, color=COLORS['medium_gray'], linestyle='--', linewidth=1.5)
ax.set_xlabel('Time $t$')
ax.set_ylabel('Hazard rate $h(t)$')
ax.set_title('Crash Hazard Rate Increases Toward $t_c$', fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)

# Right: Cumulative survival probability
ax = axes[1]
from scipy.integrate import cumulative_trapezoid
H_cumul = cumulative_trapezoid(h_full[mask_hz], t_hz[mask_hz], initial=0)
survival = np.exp(-H_cumul)
ax.plot(t_hz[mask_hz], survival, color=COLORS['purple'], linewidth=3)
ax.fill_between(t_hz[mask_hz], survival, alpha=0.2, color=COLORS['purple'])
ax.set_xlabel('Time $t$')
ax.set_ylabel('Survival probability')
ax.set_title('Probability the Bubble Survives Until $t$', fontweight='bold')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print('The hazard rate h(t) increases with log-periodic oscillations toward tc.')
print('The crash is increasingly likely but never deterministic — a probabilistic model.')

---
## Section 5: Estimation and Implementation

### Partial Linearization (Slaving)

The LPPL model has 7 parameters: $(A, B, C_1, C_2, t_c, m, \omega)$, where $C_1 = C\cos\varphi$ and $C_2 = C\sin\varphi$.

**Key insight:** For fixed nonlinear parameters $(t_c, m, \omega)$, the remaining 4 parameters $(A, B, C_1, C_2)$ enter linearly and can be solved by **OLS**:

$$\ln P(t) = A \cdot 1 + B \cdot f(t) + C_1 \cdot f(t)\cos(\omega\ln(t_c-t)) + C_2 \cdot f(t)\sin(\omega\ln(t_c-t))$$

where $f(t) = (t_c - t)^m$.

This reduces the 7D optimization to a **3D search** over $(t_c, m, \omega)$ using **Differential Evolution** (a global optimizer), with the inner problem solved exactly by OLS.

In [ ]:
# ── 5.1 LPPL Fitting Infrastructure ───────────────────────
_data_cache = {}

def download_prices(ticker, start, end):
    key = f'{ticker}_{start}_{end}'
    if key not in _data_cache:
        df = yf.download(ticker, start=start, end=end, progress=False)
        close = df['Close']
        if isinstance(close, pd.DataFrame):
            close = close.iloc[:, 0]
        _data_cache[key] = close.dropna()
    return _data_cache[key]


def _lppl_linear(t, y, tc, m, omega):
    """For fixed (tc, m, omega), solve linear params by OLS."""
    dt = tc - t
    ok = dt > 0
    t_v, y_v, dt_v = t[ok], y[ok], dt[ok]
    if len(t_v) < 10:
        return None
    f = dt_v ** m
    X = np.column_stack([np.ones(len(t_v)), f,
                         f * np.cos(omega * np.log(dt_v)),
                         f * np.sin(omega * np.log(dt_v))])
    coeffs, _, _, _ = np.linalg.lstsq(X, y_v, rcond=None)
    fitted = X @ coeffs
    ssr = float(np.sum((y_v - fitted) ** 2))
    return coeffs, ssr, fitted


def fit_lppl(prices, tc_range=None, m_range=(0.1, 0.9), omega_range=(4, 25)):
    """Fit LPPL model using differential evolution + OLS."""
    y = np.log(prices.values.astype(float))
    t = np.arange(len(y), dtype=float)
    if tc_range is None:
        tc_range = (len(y) - 5, len(y) + len(y) * 0.2)

    def objective(p):
        tc, m, omega = p
        res = _lppl_linear(t, y, tc, m, omega)
        if res is None:
            return 1e12
        coeffs, ssr, _ = res
        if coeffs[1] >= 0:
            return ssr * 100
        return ssr

    result = differential_evolution(objective, [tc_range, m_range, omega_range],
                                     seed=42, maxiter=1000, tol=1e-12,
                                     popsize=40, mutation=(0.5, 1.5), recombination=0.9)
    tc, m, omega = result.x
    coeffs, ssr, _ = _lppl_linear(t, y, tc, m, omega)
    A, B, C1, C2 = coeffs
    C = np.sqrt(C1**2 + C2**2)
    phi = np.arctan2(C2, C1)
    lam = np.exp(2 * np.pi / omega)
    sst = float(np.sum((y - np.mean(y)) ** 2))
    r2 = 1.0 - ssr / sst if sst > 0 else 0.0
    dt_all = np.maximum(tc - t, 0.01)
    f_all = dt_all ** m
    fitted_all = (A + B*f_all + C1*f_all*np.cos(omega*np.log(dt_all))
                  + C2*f_all*np.sin(omega*np.log(dt_all)))
    return {'tc': tc, 'm': m, 'omega': omega, 'A': A, 'B': B,
            'C': C, 'C1': C1, 'C2': C2, 'phi': phi,
            'lambda': lam, 'R2': r2, 'ssr': ssr,
            't': t, 'log_price': y, 'fitted_log': fitted_all, 'prices': prices}


def lppl_curve(t_arr, p):
    dt = np.maximum(p['tc'] - t_arr, 1e-6)
    f = dt ** p['m']
    return (p['A'] + p['B']*f + p['C1']*f*np.cos(p['omega']*np.log(dt))
            + p['C2']*f*np.sin(p['omega']*np.log(dt)))


print('LPPL fitting infrastructure loaded.')

### Demo 6: Fitting LPPL to Real Bubble Data

We fit the LPPL model to two classic bubble episodes using real market data:
1. **NASDAQ Dot-com bubble** (1997–2000)
2. **Bitcoin 2017 bubble**

In [ ]:
# ── Demo 6: LPPL fits to real data ────────────────────────
cases = [
    ('^IXIC', '1997-01-01', '2000-03-10', 'NASDAQ Dot-com 2000'),
    ('BTC-USD', '2017-01-01', '2017-12-16', 'Bitcoin 2017'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (ticker, start, end, title) in enumerate(cases):
    prices = download_prices(ticker, start, end)
    params = fit_lppl(prices)
    ax = axes[idx]

    ax.plot(prices.index, params['log_price'], color=COLORS['blue'], lw=1.5,
            alpha=0.7, label='Observed $\\ln P(t)$')
    ax.plot(prices.index, params['fitted_log'], color=COLORS['red'], lw=2,
            label='LPPL fit')

    tc_idx = int(np.clip(params['tc'], 0, len(prices) - 1))
    if tc_idx < len(prices):
        tc_date = prices.index[tc_idx]
    else:
        tc_date = prices.index[-1] + pd.Timedelta(days=int(params['tc'] - len(prices) + 1))
    ax.axvline(tc_date, color=COLORS['amber'], linestyle='--', lw=1.5,
               label=f'$t_c$ = {tc_date.strftime("%Y-%m-%d")}')

    ax.set_title(f'{title}\n$R^2={params["R2"]:.4f}$, $m={params["m"]:.3f}$, '
                 f'$\\omega={params["omega"]:.1f}$, $\\lambda={params["lambda"]:.2f}$',
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('Date')
    ax.set_ylabel('Log-price')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False, fontsize=9)

fig.suptitle('LPPL Fits to Historical Bubbles', fontweight='bold', y=1.01)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

for ticker, start, end, title in cases:
    prices = download_prices(ticker, start, end)
    p = fit_lppl(prices)
    print(f'\n{title}:')
    print(f'  R² = {p["R2"]:.4f}, m = {p["m"]:.3f}, omega = {p["omega"]:.2f}, lambda = {p["lambda"]:.2f}')
    print(f'  B = {p["B"]:.4f} (negative = super-exponential growth confirmed)')

---
## Section 6: The 8 Filter Conditions

A valid LPPL bubble signal must pass **8 filter conditions** ensuring physical plausibility:

| # | Condition | Financial Interpretation |
|---|-----------|------------------------|
| 1 | $0.1 \leq m \leq 0.9$ | Growth faster than exponential but not instant |
| 2 | $4 \leq \omega \leq 25$ | Log-frequency in empirically observed range |
| 3 | $B < 0$ | Price rises as $t \to t_c$ (bubble, not deflation) |
| 4 | $|C| > 0$ | Log-periodic oscillations are detectable |
| 5 | $t_c > N$ | Regime change predicted after sample end |
| 6 | $t_c < N + 0.2N$ | Not too far in the future (actionable) |
| 7 | $m|B| > 0$ | Finite-time singularity has real strength |
| 8 | $\omega/(2\pi) \geq 1$ | At least one full oscillation cycle visible |

In [ ]:
# ── 6.1 Filter conditions check ───────────────────────────
def check_filter_conditions(p, N):
    """Check 8 LPPL filter conditions."""
    tc, m, omega = p['tc'], p['m'], p['omega']
    B, C = p['B'], p['C']
    conditions = [
        ('1. 0.1 <= m <= 0.9',       0.1 <= m <= 0.9,         f'{m:.4f}'),
        ('2. 4 <= omega <= 25',       4 <= omega <= 25,        f'{omega:.2f}'),
        ('3. B < 0',                  B < 0,                   f'{B:.4f}'),
        ('4. |C| > 0',               abs(C) > 0,              f'{C:.6f}'),
        ('5. tc > N',                 tc > N,                  f'{tc:.1f}'),
        ('6. tc < N + 0.2N',          tc < N + 0.2 * N,       f'{tc:.1f}'),
        ('7. m|B| > 0',              m * abs(B) > 0,          f'{m*abs(B):.4f}'),
        ('8. omega/(2pi) >= 1',       omega / (2*np.pi) >= 1,  f'{omega/(2*np.pi):.2f}'),
    ]
    return conditions


# Check NASDAQ fit
nasdaq_prices = download_prices('^IXIC', '1997-01-01', '2000-03-10')
p_nasdaq = fit_lppl(nasdaq_prices)
conds = check_filter_conditions(p_nasdaq, len(nasdaq_prices))

print('LPPL Filter Conditions — NASDAQ Dot-com Bubble')
print('=' * 60)
n_pass = 0
for name, passed, value in conds:
    status = 'PASS' if passed else 'FAIL'
    n_pass += int(passed)
    print(f'  [{"+ " if passed else "X "}] {status}  {name:30s}  Value: {value}')
print('=' * 60)
print(f'Result: {n_pass}/8 conditions passed')

# Visual summary
fig, ax = plt.subplots(figsize=(10, 4))
labels = [f'C{i+1}' for i in range(8)]
passed_vals = [int(c[1]) for c in conds]
bar_colors = [COLORS['green'] if p else COLORS['red'] for p in passed_vals]
ax.bar(labels, passed_vals, color=bar_colors, alpha=0.8, width=0.6)
ax.set_ylim(0, 1.3)
ax.set_yticks([0, 1])
ax.set_yticklabels(['FAIL', 'PASS'])
ax.set_title(f'LPPL Filter Conditions: {n_pass}/8 Passed (NASDAQ Dot-com)',
             fontweight='bold')
for i, (name, passed, value) in enumerate(conds):
    ax.text(i, 1.1, value, ha='center', fontsize=8, color=COLORS['gray'])
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

---
## Section 7: Case Study — Historical Bubbles

LPPL has been successfully applied to major financial bubbles across different asset classes and time periods. The common pattern: super-exponential price growth with accelerating log-periodic oscillations converging on a critical time.

In [ ]:
# ── 7.1 Multi-case LPPL fits ──────────────────────────────
all_cases = [
    ('^IXIC',   '1997-01-01', '2000-03-10', 'NASDAQ Dot-com 2000'),
    ('BTC-USD', '2017-01-01', '2017-12-16', 'Bitcoin 2017'),
    ('000001.SS','2014-07-01', '2015-06-12', 'Shanghai 2015'),
    ('CL=F',    '2007-01-01', '2008-07-11', 'Oil 2008'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
colors_list = [COLORS['blue'], COLORS['red'], COLORS['green'], COLORS['purple']]

for idx, (ticker, start, end, title) in enumerate(all_cases):
    ax = axes[idx]
    try:
        prices = download_prices(ticker, start, end)
        p = fit_lppl(prices)

        # Normalize to [0, 1] for comparison
        p_norm = prices.values / prices.values[0]
        t_days = np.arange(len(prices))

        ax.plot(prices.index, p['log_price'], color=colors_list[idx], lw=1.5,
                alpha=0.7, label='Observed')
        ax.plot(prices.index, p['fitted_log'], color=COLORS['red'], lw=2,
                label='LPPL fit')
        ax.set_title(f'{title}\n$R^2={p["R2"]:.3f}$, $m={p["m"]:.2f}$, '
                     f'$\\omega={p["omega"]:.1f}$',
                     fontweight='bold', fontsize=10)
        ax.set_ylabel('Log-price')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
        ax.legend(loc='upper left', frameon=False, fontsize=9)
    except Exception as e:
        ax.text(0.5, 0.5, f'Data unavailable:\n{e}',
                transform=ax.transAxes, ha='center', va='center')
        ax.set_title(title, fontweight='bold')

fig.suptitle('LPPL Applied to Major Historical Bubbles', fontweight='bold', y=1.01)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

---
## Section 8: Bootstrap Confidence Intervals

Point estimates of $t_c$ are insufficient for risk management — we need **uncertainty quantification**.

**Residual bootstrap procedure:**
1. Fit LPPL to data, compute residuals $\hat{\varepsilon}_t = \ln P(t) - \widehat{\text{LPPL}}(t)$
2. For each of $B = 200$ bootstrap samples:
   - Resample residuals with replacement: $\varepsilon^*_t$
   - Create pseudo-data: $y^*_t = \widehat{\text{LPPL}}(t) + \varepsilon^*_t$
   - Re-estimate LPPL on $y^*$
3. Report 95% CI from bootstrap distribution of $\hat{t}_c$

In [ ]:
# ── 8.1 Bootstrap CI for NASDAQ Dot-com ───────────────────
def bootstrap_ci(prices, p0, n_boot=200):
    y = np.log(prices.values.astype(float))
    t = np.arange(len(y), dtype=float)
    resid = y - p0['fitted_log']
    tc_b, m_b, w_b = [], [], []
    for i in range(n_boot):
        rng = np.random.RandomState(i)
        y_b = p0['fitted_log'] + rng.choice(resid, len(resid), replace=True)
        def obj(p):
            res = _lppl_linear(t, y_b, p[0], p[1], p[2])
            if res is None: return 1e12
            if res[0][1] >= 0: return res[1] * 100
            return res[1]
        try:
            r = differential_evolution(obj,
                [(max(p0['tc']-30, len(y)-5), p0['tc']+30),
                 (max(0.1, p0['m']-0.15), min(0.9, p0['m']+0.15)),
                 (max(4, p0['omega']-3), min(25, p0['omega']+3))],
                seed=i, maxiter=200, tol=1e-8, popsize=15)
            tc_b.append(r.x[0]); m_b.append(r.x[1]); w_b.append(r.x[2])
        except Exception:
            pass
    def ci(a):
        a = np.array(a)
        return (np.percentile(a, 2.5), np.percentile(a, 97.5)) if len(a) > 10 else (np.nan, np.nan)
    return {'tc_ci': ci(tc_b), 'm_ci': ci(m_b), 'omega_ci': ci(w_b),
            'tc_boot': np.array(tc_b), 'm_boot': np.array(m_b), 'omega_boot': np.array(w_b)}


print('Running 200 bootstrap replications for NASDAQ Dot-com...')
p_nasdaq = fit_lppl(nasdaq_prices)
boot = bootstrap_ci(nasdaq_prices, p_nasdaq, n_boot=200)
print(f'Done. Valid fits: {len(boot["tc_boot"])}')

# Plot bootstrap distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, param, label, color in [
    (axes[0], boot['tc_boot'], '$t_c$', COLORS['blue']),
    (axes[1], boot['m_boot'],  '$m$',   COLORS['green']),
    (axes[2], boot['omega_boot'], '$\\omega$', COLORS['purple']),
]:
    ax.hist(param, bins=30, color=color, alpha=0.6, density=True)
    lo, hi = np.percentile(param, [2.5, 97.5])
    ax.axvline(lo, color=COLORS['amber'], lw=1.5, ls=':')
    ax.axvline(hi, color=COLORS['amber'], lw=1.5, ls=':')
    ax.set_title(f'Bootstrap Distribution of {label}', fontweight='bold')
    ax.set_xlabel(label)
    ax.text(0.5, 0.95, f'95% CI: [{lo:.2f}, {hi:.2f}]',
            transform=ax.transAxes, ha='center', va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

fig.suptitle('Bootstrap Confidence Intervals — NASDAQ Dot-com', fontweight='bold', y=1.02)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print(f'\n95% CI for tc: [{boot["tc_ci"][0]:.1f}, {boot["tc_ci"][1]:.1f}] trading days')
print(f'95% CI for m:  [{boot["m_ci"][0]:.3f}, {boot["m_ci"][1]:.3f}]')
print(f'95% CI for omega: [{boot["omega_ci"][0]:.2f}, {boot["omega_ci"][1]:.2f}]')

---
## Section 9: LPPLS Confidence Indicator

The **LPPLS Confidence Indicator** (Sornette et al., 2015) provides a real-time bubble risk assessment:

1. At each date $t$, fit LPPL over **multiple windows** ending at $t$
2. Count the fraction of fits that pass all 8 filter conditions
3. Traffic-light interpretation:
   - **Green** (CI < 0.3): No significant bubble signal
   - **Amber** (0.3 ≤ CI < 0.6): Emerging bubble signal
   - **Red** (CI ≥ 0.6): Strong bubble warning

In [ ]:
# ── 9.1 LPPLS Confidence Indicator — BTC 2021 ─────────────
btc_ci = download_prices('BTC-USD', '2020-09-01', '2021-11-15')
log_btc = np.log(btc_ci.values.astype(float))
N_btc = len(log_btc)

ci_windows = [60, 90, 120, 150, 180]
min_win = max(ci_windows)
step = 10

def _fit_lppl_array(t, y, seed=42):
    """Fit LPPL to array data (not Series)."""
    N = len(y)
    def objective(p):
        tc, m, omega = p
        res = _lppl_linear(t, y, tc, m, omega)
        if res is None: return 1e12
        if res[0][1] >= 0: return res[1] * 100
        return res[1]
    result = differential_evolution(objective,
        [(N - 5, N + N * 0.2), (0.1, 0.9), (4, 25)],
        seed=seed, maxiter=300, tol=1e-8, popsize=20)
    tc, m, omega = result.x
    res = _lppl_linear(t, y, tc, m, omega)
    if res is None: return None
    A, B, C1, C2 = res[0]
    C = np.sqrt(C1**2 + C2**2)
    return {'tc': tc, 'm': m, 'omega': omega, 'B': B, 'C': C}


eval_indices = list(range(min_win, N_btc, step))
ci_dates = []
ci_values = []

print(f'Computing LPPLS CI at {len(eval_indices)} dates (step={step})...')
for count, end_idx in enumerate(eval_indices):
    n_valid = 0
    n_total = 0
    for w in ci_windows:
        if end_idx < w:
            continue
        t_w = np.arange(w, dtype=float)
        y_w = log_btc[end_idx - w:end_idx]
        n_total += 1
        try:
            p_ci = _fit_lppl_array(t_w, y_w, seed=42)
            if p_ci is not None:
                conds = check_filter_conditions(p_ci, w)
                if sum(c[1] for c in conds) == 8:
                    n_valid += 1
        except Exception:
            pass
    ci_val = n_valid / max(n_total, 1)
    ci_dates.append(btc_ci.index[end_idx - 1])
    ci_values.append(ci_val)
    if (count + 1) % 10 == 0:
        print(f'  {count + 1}/{len(eval_indices)} done')

ci_series = pd.Series(ci_values, index=ci_dates)
print(f'Done. CI computed at {len(ci_series)} dates.')

In [ ]:
# ── 9.2 Plot CI with traffic-light colors ──────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8),
                                gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(btc_ci.index, btc_ci.values, color=COLORS['blue'], lw=1.5)
ax1.set_title('BTC-USD Price with LPPLS Confidence Indicator', fontweight='bold')
ax1.set_ylabel('Price (USD)')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

for i in range(len(ci_series) - 1):
    ci_val = ci_series.iloc[i]
    if ci_val >= 0.6:
        color = COLORS['red']
    elif ci_val >= 0.3:
        color = COLORS['amber']
    else:
        color = COLORS['green']
    ax2.bar(ci_series.index[i], ci_val, width=step + 2, color=color, alpha=0.7)

ax2.axhline(0.3, color=COLORS['amber'], ls='--', lw=1, alpha=0.8)
ax2.axhline(0.6, color=COLORS['red'], ls='--', lw=1, alpha=0.8)
ax2.set_ylim(0, 1)
ax2.set_title('LPPLS Confidence Indicator', fontweight='bold')
ax2.set_ylabel('CI')
ax2.set_xlabel('Date')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b\n%Y'))

legend_elements = [
    mpatches.Patch(facecolor=COLORS['green'], alpha=0.7, label='CI < 0.3 (No signal)'),
    mpatches.Patch(facecolor=COLORS['amber'], alpha=0.7, label='0.3 ≤ CI < 0.6 (Emerging)'),
    mpatches.Patch(facecolor=COLORS['red'],   alpha=0.7, label='CI ≥ 0.6 (Strong warning)'),
]
ax2.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.25),
           ncol=3, frameon=False)

fig.patch.set_alpha(0)
plt.tight_layout()
plt.show()

print(f'Maximum CI: {ci_series.max():.2f} on {ci_series.idxmax().strftime("%Y-%m-%d")}')
print(f'Dates with CI >= 0.6: {(ci_series >= 0.6).sum()}')
print(f'Dates with CI >= 0.3: {(ci_series >= 0.3).sum()}')

---
## Section 10: Summary and Key Takeaways

### Core Concepts

| Topic | Key Idea | Connection |
|-------|----------|------------|
| **Super-exponential growth** | Price growth accelerates toward $t_c$ | Positive feedback loops |
| **Ising model** | Spins on a lattice with herding interactions | Market = interacting agents |
| **Phase transition** | Order → disorder at $T_c$ | Bubble → regime break |
| **Susceptibility** | $\chi \to \infty$ at critical point | Maximum market fragility |
| **Discrete scale invariance** | Preferred scaling ratio $\lambda$ | Hierarchical market structure |
| **Log-periodic oscillations** | Accelerating oscillations in log-time | Rounds of trader hesitation |
| **LPPL equation** | Power law + log-periodic modulation | The bubble signature |
| **Partial linearization** | 7D → 3D search + OLS | Efficient estimation |
| **8 filter conditions** | Parameter bounds for valid signal | Physical plausibility |
| **Bootstrap CI** | Residual resampling → $t_c$ uncertainty | Risk quantification |
| **LPPLS CI** | Multi-window rolling indicator | Real-time monitoring |

### Notation Clarification

- $T_c$ = critical **temperature** (Ising model, physics parameter)
- $t_c$ = critical **time** (LPPL model, calendar date of regime change)
- Same mathematical framework, different domains

### When to Use LPPL

- **Yes:** Endogenous bubbles (dot-com, crypto, housing) driven by herding and positive feedback
- **No:** Exogenous shocks (COVID, geopolitical events) — LPPL correctly produces no signal
- **Caution:** LPPL estimates $t_c$ as the *most probable* time for a regime change, not a guaranteed crash date

### Next Steps

- **Seminar:** Hands-on LPPL fitting, validation, and risk management
- **Advanced:** Multi-scale LPPLS, Lomb-Scargle spectral analysis, nonlinear model extensions